# Set Environment

In [2]:
# Install required libraries and mount Google Drive.
# Run this cell once at the start of each session.

!pip install gspread -q

from google.colab import drive, auth
from google.auth import default
import gspread
import pandas as pd
import numpy as np
import time
import os

drive.mount('/content/drive', force_remount=True)

# Authenticate using your Google account
# This uses the same account you used to mount Drive
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

print("Setup complete.")
print("Google Sheets authenticated successfully.")

Mounted at /content/drive
Setup complete.
Google Sheets authenticated successfully.


# Configurations

In [5]:
# All settings for this notebook live here.
# Update these values if paths or sheet names change.
# This is the only cell you should need to edit.

# =============================================================================
# FILE PATHS
# =============================================================================
DASHBOARD_CSV_PATH = '/content/drive/Shareddrives/essentis_intern_drive/data/processed/dashboard_master.csv'

# =============================================================================
# GOOGLE SHEETS SETTINGS
# To find your Spreadsheet ID:
#   1. Open your Google Sheet in the browser
#   2. The URL looks like:
#      https://docs.google.com/spreadsheets/d/SPREADSHEET_ID/edit
#   3. Copy the long string between /d/ and /edit
# =============================================================================
SPREADSHEET_ID      = "1xGHnwMXzn9ISHSSGKOB0h53whO3h6g16ifmZ1f1uSR4"  # replace with your actual ID
SHEET_TAB_NAME      = "dashboard_master"           # tab name inside the spreadsheet
FREEZE_HEADER_ROWS  = 1                            # number of header rows to freeze
BATCH_SIZE          = 5000                         # rows per API write call

print("Configuration loaded.")
print(f"Source CSV:    {DASHBOARD_CSV_PATH}")
print(f"Sheet tab:     {SHEET_TAB_NAME}")
print(f"Batch size:    {BATCH_SIZE} rows per call")

Configuration loaded.
Source CSV:    /content/drive/Shareddrives/essentis_intern_drive/data/processed/dashboard_master.csv
Sheet tab:     dashboard_master
Batch size:    5000 rows per call


# Load Dash CSV

In [7]:
# Load dashboard_master.csv and run basic validation checks
# before attempting to write to Google Sheets.
# Catches common issues early so the sheet is not partially overwritten.

print("Loading dashboard_master.csv...")

if not os.path.exists(DASHBOARD_CSV_PATH):
    raise FileNotFoundError(
        f"dashboard_master.csv not found at {DASHBOARD_CSV_PATH}\n"
        f"Run the cleaning pipeline notebook first to generate it."
    )

dashboard_df = pd.read_csv(DASHBOARD_CSV_PATH)

print(f"Shape:          {dashboard_df.shape}")
print(f"Rows:           {len(dashboard_df)}")
print(f"Columns:        {len(dashboard_df.columns)}")
print(f"\nColumn list:")
for col in dashboard_df.columns:
    print(f"  {col}")

# Basic validation
issues = []
if len(dashboard_df) == 0:
    issues.append("DataFrame is empty")
if len(dashboard_df.columns) == 0:
    issues.append("No columns found")
if 'review_text' not in dashboard_df.columns:
    issues.append("Missing expected column: review_text")
if 'overall_experience' not in dashboard_df.columns:
    issues.append("Missing expected column: overall_experience")

if issues:
    print(f"\nWARNING: Validation issues found:")
    for issue in issues:
        print(f"  - {issue}")
    print("Review the issues above before proceeding.")
else:
    print(f"\nValidation passed. Ready to upload.")

Loading dashboard_master.csv...
Shape:          (839, 59)
Rows:           839
Columns:        59

Column list:
  batch_id
  author
  data_source
  source
  review_date
  overall_experience
  review
  instructors
  curriculum
  job_assistance
  review_text
  review_text_translated
  course_format
  course
  role
  verified
  verification_source
  link
  text_for_analysis
  review_year
  review_month
  verified_label
  aspect_teaching_quality
  aspect_curriculum
  aspect_career_support
  aspect_community
  aspect_course_pace
  aspect_value_for_money
  aspect_online_experience
  aspect_support_and_staff
  aspect_hardware_and_setup
  aspect_outcomes
  motivation_career_change
  motivation_job_outcomes
  motivation_curriculum_quality
  motivation_flexibility
  motivation_reputation
  motivation_price_value
  motivation_community
  motivation_speed
  semantic_best_score
  semantic_best_query
  semantic_theme
  dominant_topic
  topic_weight
  sentiment_compound
  sentiment_pos
  sentiment_neg

# Prep Sheets Formats

In [8]:
# Convert the DataFrame to a format safe for the Google Sheets API.
# Google Sheets cannot handle NaN, pd.NA, numpy types, or pandas
# nullable integers. Everything must be native Python types.

def clean_value_for_sheets(val: any) -> any:
    """
    Convert a single DataFrame cell value to a Google Sheets safe type.

    Google Sheets API only accepts native Python str, int, float, and bool.
    This function converts all pandas and numpy types to one of those four,
    replacing all null-like values with empty string.

    Args:
        val (any): A single cell value from a pandas DataFrame.

    Returns:
        any: A native Python str, int, float, bool, or empty string.
    """
    # Handle None
    if val is None:
        return ""

    # Handle pandas NA and NaT
    try:
        if pd.isna(val):
            return ""
    except (TypeError, ValueError):
        pass

    # Handle numpy integers
    if isinstance(val, np.integer):
        return int(val)

    # Handle numpy floats
    if isinstance(val, np.floating):
        if np.isnan(val):
            return ""
        return float(val)

    # Handle numpy booleans
    if isinstance(val, np.bool_):
        return bool(val)

    # Handle everything else as string
    s = str(val)
    if s in ('nan', 'NaN', 'NaT', 'None', '<NA>', 'NA', 'pd.NA'):
        return ""
    return s


def dataframe_to_sheets_rows(df: pd.DataFrame) -> list:
    """
    Convert a full DataFrame to a list of lists for Google Sheets upload.

    The first row contains column headers. All subsequent rows contain
    data values cleaned to native Python types via clean_value_for_sheets().

    Args:
        df (pd.DataFrame): The DataFrame to convert.

    Returns:
        list: List of lists where row 0 is headers and rows 1+ are data.
              All values are Google Sheets API safe native Python types.
    """
    rows = [df.columns.tolist()]
    for _, row in df.iterrows():
        rows.append([clean_value_for_sheets(v) for v in row])
    return rows


print("Preparing data for Google Sheets API...")
rows = dataframe_to_sheets_rows(dashboard_df)

# Spot check a few values to confirm conversion worked
print(f"Total rows prepared: {len(rows)} (including 1 header row)")
print(f"Columns:             {len(rows[0])}")
print(f"\nSample header:  {rows[0][:5]}")
print(f"Sample data row:{rows[1][:5]}")

# Check for any remaining problematic types
problem_count = 0
for i, row in enumerate(rows[1:6]):  # check first 5 data rows
    for j, val in enumerate(row):
        if not isinstance(val, (str, int, float, bool)):
            print(f"  WARNING: row {i+1}, col {j} has type {type(val)}: {val}")
            problem_count += 1

if problem_count == 0:
    print("\nType check passed. All values are Sheets-safe.")
else:
    print(f"\nWARNING: {problem_count} values may need attention.")

Preparing data for Google Sheets API...
Total rows prepared: 840 (including 1 header row)
Columns:             59

Sample header:  ['batch_id', 'author', 'data_source', 'source', 'review_date']
Sample data row:['613809945077.0', 'Gabriel Gomez ', 'clean', 'course_report', '20230907.0']

Type check passed. All values are Sheets-safe.


# Connect to Sheets

In [9]:
# Connect to the target spreadsheet and get or create the target tab.
# Re-authenticates in case the Colab session has timed out.

print("Connecting to Google Sheets...")

# Re-authenticate to handle session timeouts
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open the spreadsheet
try:
    spreadsheet = gc.open_by_key(SPREADSHEET_ID)
    print(f"Opened spreadsheet: {spreadsheet.title}")
except gspread.exceptions.SpreadsheetNotFound:
    raise ValueError(
        f"Spreadsheet not found. Check your SPREADSHEET_ID.\n"
        f"Current value: {SPREADSHEET_ID}\n"
        f"Expected format: a long alphanumeric string from the Google Sheets URL."
    )
except gspread.exceptions.APIError as e:
    raise ValueError(
        f"Google Sheets API error: {e}\n"
        f"Make sure the spreadsheet is shared with your Google account."
    )

# Get or create the target tab
try:
    worksheet = spreadsheet.worksheet(SHEET_TAB_NAME)
    print(f"Found existing tab: {SHEET_TAB_NAME}")
    print(f"Current tab size: {worksheet.row_count} rows x {worksheet.col_count} cols")
except gspread.exceptions.WorksheetNotFound:
    worksheet = spreadsheet.add_worksheet(
        title=SHEET_TAB_NAME,
        rows=len(rows) + 10,
        cols=len(rows[0]) + 5
    )
    print(f"Created new tab: {SHEET_TAB_NAME}")

print(f"\nReady to write {len(rows) - 1} data rows to {SHEET_TAB_NAME}.")

Connecting to Google Sheets...
Opened spreadsheet: dashboard_master
Found existing tab: dashboard_master
Current tab size: 842 rows x 58 cols

Ready to write 839 data rows to dashboard_master.


# Check

In [10]:
# Read back a sample from the sheet to confirm the data was written correctly.
# Checks row count, column count, and a sample of values.

print("Verifying upload...")

# Read back the first 3 rows from the sheet
sample = worksheet.get('A1:E3')
print(f"\nSample from sheet (first 3 rows, first 5 columns):")
for i, row in enumerate(sample):
    label = "Header" if i == 0 else f"Row {i}"
    print(f"  {label}: {row}")

# Confirm row count matches
sheet_row_count = len(worksheet.get_all_values())
expected_rows   = len(dashboard_df) + 1  # +1 for header
match = sheet_row_count == expected_rows

print(f"\nRow count check:")
print(f"  Expected: {expected_rows} ({len(dashboard_df)} data + 1 header)")
print(f"  In sheet: {sheet_row_count}")
print(f"  Match:    {'✓' if match else '✗ mismatch — check for API errors above'}")

sheet_url = f"https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit"
print(f"\n=== GOOGLE SHEETS UPDATE COMPLETE ===")
print(f"Spreadsheet: {spreadsheet.title}")
print(f"Tab:         {SHEET_TAB_NAME}")
print(f"URL:         {sheet_url}")
print(f"\nLooker Studio will pick up the new data on its next scheduled refresh.")
print(f"To refresh immediately: open your Looker report and click the refresh icon.")

Verifying upload...

Sample from sheet (first 3 rows, first 5 columns):
  Header: ['batch_id', 'author', 'data_source', 'source', 'review_date']
  Row 1: ['613809945077', 'Gabriel Gomez', 'clean', 'course_report', '20230907']
  Row 2: ['613809945077', 'Andrej S.', 'clean', 'course_report', '20230905']

Row count check:
  Expected: 840 (839 data + 1 header)
  In sheet: 840
  Match:    ✓

=== GOOGLE SHEETS UPDATE COMPLETE ===
Spreadsheet: dashboard_master
Tab:         dashboard_master
URL:         https://docs.google.com/spreadsheets/d/1xGHnwMXzn9ISHSSGKOB0h53whO3h6g16ifmZ1f1uSR4/edit

Looker Studio will pick up the new data on its next scheduled refresh.
To refresh immediately: open your Looker report and click the refresh icon.
